# Pipeline CBR + LLM para explicaciones XAI - version usuario final

Esta version esta pensada para ejecutar el experimento con un usuario real. La interfaz genera la explicacion final

## 1. Imports y rutas

In [1]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd
from IPython.display import Image as DisplayImage, Markdown, display


def find_project_root(start: Path, project_name="TFM-Personalized-XAI") -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if candidate.name == project_name and (candidate / "evaluacion_online").exists():
            return candidate
        if (candidate / "evaluacion_online").exists() and (candidate / "base_de_casos").exists():
            return candidate
    raise FileNotFoundError(f"No encuentro la raiz del proyecto {project_name}")


PROJECT_ROOT = find_project_root(Path.cwd())
for path in [PROJECT_ROOT, PROJECT_ROOT / "evaluacion_online"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import llm_cbr_pipeline
importlib.reload(llm_cbr_pipeline)

from llm_cbr_pipeline import (
    CBRRetriever,
    DEFAULT_CASE_BASE,
    DEFAULT_DESCRIPTIONS,
    DEFAULT_OUTPUT_DIR,
    METHOD_LABEL,
    OPTION_TO_METHOD,
    PipelineResult,
    build_problem_representation,
    build_prompt,
    build_solution_representation,
    generate_with_ollama,
    get_xai_description,
    is_missing,
    save_result,
    select_recommended_solution,
)

CASE_BASE = DEFAULT_CASE_BASE
DESCRIPTIONS = DEFAULT_DESCRIPTIONS
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

print("Proyecto:", PROJECT_ROOT)
print("Base de casos:", CASE_BASE)
print("Descripciones XAI:", DESCRIPTIONS)
print("Carpeta de salida:", OUTPUT_DIR)


Proyecto: /Users/haojie/PycharmProjects/TFM-Personalized-XAI
Base de casos: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/base_de_casos/cbr_case_base_outputs/case_base_double_full.csv
Descripciones XAI: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/generacion_descripcion_XAI/resultados_descripciones_xai/descripciones_por_imagen_xai.csv
Carpeta de salida: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/evaluacion_online/resultados


## 2. Configuracion



In [2]:
RUN_OLLAMA = False
OLLAMA_MODEL = "qwen2.5vl:32b"
K_NEIGHBORS = 5
SAME_IMAGE_ONLY = False
MIN_SIMILARITY = 0.60

IMAGES_DIR = PROJECT_ROOT / "imagenes"


def project_image_path(folder, file_name):
    return IMAGES_DIR / str(folder) / str(file_name)


def resolve_description_image_path(row):
    raw_path = Path(str(row.get("image_path", "")))
    project_path = project_image_path(raw_path.parent.name, raw_path.name)
    if project_path.exists():
        return project_path
    folder = row.get("folder", "")
    file_name = row.get("file_name", "")
    fallback_path = project_image_path(folder, file_name)
    if fallback_path.exists():
        return fallback_path
    return project_path


IMAGE_METADATA_PATH = PROJECT_ROOT / "base_de_casos/cbr_case_base_outputs/image_metadata_template.csv"
image_map_df = pd.read_csv(IMAGE_METADATA_PATH).rename(
    columns={"model_predicted_class": "class_label"}
)
image_map_df["description_case_id"] = image_map_df["image_id"].astype(int).map(lambda x: f"image{x:02d}")
image_map_df["original_image_path"] = image_map_df["description_case_id"].map(
    lambda case_id: project_image_path("original", f"{case_id}.jpg")
)
image_map_df = image_map_df[
    ["image_id", "image_label", "class_label", "description_case_id", "original_image_path"]
]
IMAGE_ID_TO_DESCRIPTION_CASE = dict(zip(image_map_df["image_id"], image_map_df["description_case_id"]))
IMAGE_ID_TO_LABEL = {
    int(row.image_id): f"{int(row.image_id)} - {row.class_label} ({row.image_label})"
    for row in image_map_df.itertuples(index=False)
}

DEMO_IMAGE_ID = 3
DESCRIPTION_CASE_ID = IMAGE_ID_TO_DESCRIPTION_CASE[DEMO_IMAGE_ID]


## 3. Generador de explicacion final

Selecciona la imagen y el perfil del usuario. Al pulsar **Generar explicacion**, el notebook mostrara solo la explicacion final para el usuario.


In [3]:
import html
import ipywidgets as widgets
from IPython.display import clear_output

def make_checkbox_group(options, selected=None):
    selected = set(selected or [])
    return [
        widgets.Checkbox(value=option in selected, description=option, indent=False)
        for option in options
    ]
retriever = CBRRetriever(CASE_BASE)
descriptions_df = pd.read_csv(DESCRIPTIONS)
available_image_ids = sorted(retriever.df["image_id"].dropna().astype(int).unique())
image_selector_options = [
    (IMAGE_ID_TO_LABEL.get(image_id, f"Imagen {image_id}"), image_id)
    for image_id in available_image_ids
    if image_id in IMAGE_ID_TO_DESCRIPTION_CASE
]

age_w = widgets.Dropdown(
    description="Edad",
    options=["18-24", "25-34", "35-44", "45-54", "55-64", "65 o mas"],
    value="25-34",
)
education_w = widgets.Dropdown(
    description="Estudios",
    options=["Bachillerato/FP", "Grado", "Master", "Doctorado", "Prefiero no contestar"],
    value="Master",
)
occupation_options = [
    "Estudiante",
    "Investigador/a (academico)",
    "Docente / profesor/a",
    "Profesional del sector tecnologico / IA / datos",
    "Profesional de otro sector",
    "Desempleado/a / en busqueda",
    "Prefiero no contestar",
]
occupation_checks = make_checkbox_group(
    occupation_options,
    selected=["Investigador/a (academico)"],
)
occupation_box = widgets.VBox(occupation_checks)
ai_level_w = widgets.IntSlider(description="IA", min=1, max=5, value=4)
domain_level_w = widgets.IntSlider(description="Dominio", min=1, max=5, value=3)


def selected_checkbox_values(checkboxes):
    return [checkbox.description for checkbox in checkboxes if checkbox.value]

CBR_INFERRED_PREFERENCE_FIELDS = [
    "preferred_response_length",
    "preferred_technical_level",
    "preferred_format",
    "preferred_explanation_types_raw",
    "main_goals_raw",
    "perceived_error_impact",
]
image_w = widgets.Dropdown(
    description="Imagen",
    options=image_selector_options,
    value=DEMO_IMAGE_ID if DEMO_IMAGE_ID in available_image_ids else image_selector_options[0][1],
    layout=widgets.Layout(width="650px"),
)
use_ollama_w = widgets.Checkbox(description="Usar Ollama", value=RUN_OLLAMA)
generate_button = widgets.Button(description="Generar explicacion", button_style="primary")
preview_title = widgets.HTML()
preview_original_image = widgets.Image(layout=widgets.Layout(width="110px"))
preview_xai_title = widgets.HTML()
preview_xai_image = widgets.Image(layout=widgets.Layout(width="170px"))
preview_box = widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Imagen principal</b>"), preview_original_image]),

])

display(Markdown("### Perfil e imagen"))
display(widgets.VBox([
    widgets.HBox([image_w, use_ollama_w]),
    preview_box,
    widgets.HBox([age_w, education_w]),
    widgets.HTML("<b>Ocupacion</b>"),
    occupation_box,
    widgets.HBox([ai_level_w, domain_level_w]),
    generate_button,
]))

result_box = widgets.VBox()
feedback_box = widgets.VBox()
display(result_box, feedback_box)

interactive_state = {"result": None, "neighbor": None, "iteration": 0, "is_generating": False, "session_history": [], "rejected_methods": []}
ALTERNATIVE_POOL_SIZE = len(retriever.df)

def html_title(text, level=3):
    return widgets.HTML(f"<h{level}>{html.escape(str(text))}</h{level}>")

def html_text(text):
    return widgets.HTML(
        "<div style='white-space:pre-wrap; max-width:900px; line-height:1.45'>"
        f"{html.escape(str(text))}"
        "</div>"
    )

def solution_widget(solution):
    if not solution:
        return widgets.HTML("<i>Solucion CBR no disponible.</i>")
    components = solution.get("explanation_components", {})
    final_output = solution.get("final_output", {})
    rows = [
        ("Basado en features", components.get("based_on_features", {}).get("include"), components.get("based_on_features", {}).get("description")),
        ("Basado en instancias", components.get("based_on_instances", {}).get("include"), components.get("based_on_instances", {}).get("description")),
        ("Contraejemplos", components.get("counterexamples", {}).get("include"), components.get("counterexamples", {}).get("description")),
        ("Limitaciones / dudas", components.get("limitations", {}).get("include"), components.get("limitations", {}).get("description")),
    ]
    rows_html = "".join(
        "<tr>"
        f"<td>{html.escape(str(name))}</td>"
        f"<td>{html.escape(str(include))}</td>"
        f"<td>{html.escape(str(description or ''))}</td>"
        "</tr>"
        for name, include, description in rows
    )
    params = (
        f"<p><b>Metodo:</b> {html.escape(str(solution.get('recommended_method')))} · "
        f"<b>K_instancias:</b> {html.escape(str(solution.get('k_instances')))} · "
        #f"<b>K_contraejemplos:</b> {html.escape(str(solution.get('k_counterexamples')))}</p>"
        f"<p><b>Longitud:</b> {html.escape(str(final_output.get('length')))} · "
        f"<b>Nivel tecnico:</b> {html.escape(str(final_output.get('technical_level')))} · "
        f"<b>Estructura:</b> {html.escape(str(final_output.get('structure')))}</p>"
    )
    return widgets.HTML(
        "<div style='max-width:980px'>"
        + params
        + "<table style='border-collapse:collapse'>"
        + "<thead><tr><th>Componente</th><th>Incluir</th><th>Descripcion</th></tr></thead>"
        + f"<tbody>{rows_html}</tbody></table></div>"
    )

def dataframe_widget(df):
    return widgets.HTML(
        df.to_html(index=False, border=0, classes="dataframe", justify="left")
    )

def image_widget(path, width="220px"):
    image_path = Path(path)
    fmt = image_path.suffix.lower().lstrip(".").replace("jpg", "jpeg")
    return widgets.Image(
        value=image_path.read_bytes(),
        format=fmt,
        layout=widgets.Layout(width=width),
    )

def build_interactive_query():
    return {
        "image_id": image_w.value,
        "age_range": age_w.value,
        "education_level": education_w.value,
        "occupation_raw": ", ".join(selected_checkbox_values(occupation_checks)),
        "ai_knowledge_level": ai_level_w.value,
        "domain_knowledge_level": domain_level_w.value,
    }


def complete_query_with_recovered_preferences(query, neighbor):
    completed_query = dict(query)
    for field in CBR_INFERRED_PREFERENCE_FIELDS:
        value = neighbor.get(field, None)
        if pd.notna(value):
            completed_query[field] = value
    return completed_query

def resolve_image_path(row):
    return resolve_description_image_path(row)


def get_description_row(description_case_id, method):
    case_rows = descriptions_df[descriptions_df["case_id"].astype(str) == str(description_case_id)]
    method_rows = case_rows[case_rows["method"] == method]
    if method_rows.empty:
        return None
    return method_rows.iloc[0]


def build_original_image_widgets(description_case_id):
    row = get_description_row(description_case_id, "original")
    if row is None:
        return []
    original_path = resolve_image_path(row)
    if not original_path.exists():
        return []
    return [
        widgets.HTML("<b>Imagen principal original</b>"),
      
    ]


def build_xai_image_widgets(description_case_id, method, title="Visualizacion XAI seleccionada"):
    if method == "none":
        return [widgets.HTML("<i>No se ha seleccionado una explicacion XAI visual.</i>")]
    row = get_description_row(description_case_id, method)
    if row is None:
        return []
    xai_path = resolve_image_path(row)
    if not xai_path.exists():
        return []
    return [
        widgets.HTML(f"<b>{html.escape(title)}: {html.escape(METHOD_LABEL.get(method, method))}</b>"),
        image_widget(xai_path, width="220px"),
    ]


def build_case_gallery(cases_df, max_items=5, include_xai=True, empty_message="No se pudieron cargar imagenes."):
    cards = []
    for _, case in cases_df.head(max_items).iterrows():
        image_id = case.get("image_id")
        if pd.isna(image_id):
            continue
        image_id = int(image_id)
        description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE.get(image_id)
        if description_case_id is None:
            continue
        method = OPTION_TO_METHOD.get(str(case.get("selected_option")), "none")
        original_widgets = build_original_image_widgets(description_case_id)
        xai_widgets = build_xai_image_widgets(description_case_id, method, title="XAI del caso")
        similarity = case.get("similarity", 0)
        try:
            similarity_text = f"{float(similarity):.3f}"
        except Exception:
            similarity_text = str(similarity)
        card_children = [
            widgets.HTML(
                "<div style='font-size:12px; line-height:1.35'>"
                f"<b>{html.escape(str(case.get('case_id', '')))}</b><br>"
                f"usuario: {html.escape(str(case.get('user_id', '')))}<br>"
                f"image_id: {html.escape(str(image_id))}<br>"
                f"clase: {html.escape(str(case.get('model_predicted_class', '')))}<br>"
                f"opcion: {html.escape(str(case.get('selected_option', '')))}<br>"
                f"sim: {html.escape(similarity_text)}"
                "</div>"
            )
        ]
        if original_widgets:
            card_children.append(original_widgets[-1])
        if include_xai and len(xai_widgets) > 1:
            card_children.append(widgets.HTML(f"<span style='font-size:12px'>{html.escape(METHOD_LABEL.get(method, method))}</span>"))
            card_children.append(xai_widgets[-1])
        cards.append(widgets.VBox(
            card_children,
            layout=widgets.Layout(
                width="180px",
                border="1px solid #ddd",
                padding="6px",
                margin="0 8px 8px 0",
            ),
        ))
    if not cards:
        return widgets.HTML(f"<i>{html.escape(empty_message)}</i>")
    return widgets.HBox(cards, layout=widgets.Layout(flex_flow="row wrap"))


def build_recovered_images_gallery(neighbors, max_items=5):
    return build_case_gallery(
        neighbors,
        max_items=max_items,
        include_xai=True,
        empty_message="No se pudieron cargar imagenes de los vecinos recuperados.",
    )



def update_selection_preview(change=None):
    selected_image_id = image_w.value
    selected_case_id = IMAGE_ID_TO_DESCRIPTION_CASE[selected_image_id]
    original_row = get_description_row(selected_case_id, "original")
    preview_title.value = (
        f"<b>Previsualizacion seleccionada</b><br>"
        f"<code>image_id</code>: {selected_image_id} · "
        f"imagen: <code>{selected_case_id}</code><br>"
        "Explicacion: <b>recomendada automaticamente por CBR</b>"
    )
    preview_original_image.value = b""
    preview_original_image.format = "png"
    preview_xai_title.value = ""
    preview_xai_image.value = b""
    preview_xai_image.format = "png"
    if original_row is not None:
        original_path = resolve_image_path(original_row)
        if original_path.exists():
            preview_original_image.value = original_path.read_bytes()
            preview_original_image.format = original_path.suffix.lower().lstrip(".").replace("jpg", "jpeg")
        else:
            preview_title.value += "<br><span style='color:#b00020'>La ruta de la imagen original no existe.</span>"
    else:
        preview_title.value += "<br><span style='color:#b00020'>No hay imagen original asociada a esta seleccion.</span>"

image_w.observe(update_selection_preview, names="value")
update_selection_preview()

def fallback_explanation(method_label, xai_text):
    return (
        f"El metodo XAI usado es {method_label}. "
        "La visualizacion resalta las zonas de la imagen que mas influyen en la decision del modelo, "
        "por lo que permite comprobar si la prediccion se apoya en rasgos relevantes del objeto clasificado "
        "y no en elementos secundarios del fondo.\n\n"
        "Te convence esta explicacion?"
    )


def build_final_user_prompt(query, recommended_option, recommended_method, xai_description, neighbor=None, requested_change=None):
    content = build_problem_representation(query)["content_layer"]
    solution = build_solution_representation(query, recommended_option, recommended_method)
    final_output = solution.get("final_output", {})
    class_name = content.get("model_predicted_class") or "la clase predicha"
    neighbor = neighbor if neighbor is not None else {}

    def value_or_no_indicado(source, field):
        try:
            value = source.get(field, None)
        except AttributeError:
            value = None
        if is_missing(value) or str(value).strip() == "":
            return "no indicado"
        return value

    change_block = ""
    if requested_change:
        change_block = (
            "\nEl usuario no quedo convencido con la explicacion anterior. "
            f"Ahora necesita: {requested_change}\n"
        )

    return f"""Eres un modulo LLM para explicar una clasificacion de imagen con XAI.
Genera una explicacion final en espanol para el usuario.

Contexto interno, no lo muestres como apartados:
- Clase predicha por el modelo: {class_name}
- Opcion recomendada por el CBR: {recommended_option}
- Metodo XAI seleccionado: {METHOD_LABEL.get(recommended_method, recommended_method)}
- Respuesta/preferencia recuperada del caso similar:
  - Longitud preferida: {value_or_no_indicado(neighbor, "preferred_response_length")}
  - Nivel tecnico preferido: {value_or_no_indicado(neighbor, "preferred_technical_level")}
  - Formato preferido: {value_or_no_indicado(neighbor, "preferred_format")}
  - Tipos de explicacion preferidos: {value_or_no_indicado(neighbor, "preferred_explanation_types_raw")}
  - Objetivos principales: {value_or_no_indicado(neighbor, "main_goals_raw")}
  - Impacto percibido del error: {value_or_no_indicado(neighbor, "perceived_error_impact")}
  - Satisfaccion/confianza/comprension del usuario recuperado: {value_or_no_indicado(neighbor, "satisfaction")}/{value_or_no_indicado(neighbor, "confidence")}/{value_or_no_indicado(neighbor, "understanding")}
  - Comentario libre del usuario recuperado: {value_or_no_indicado(neighbor, "free_comment")}
- Salida final inferida desde la solucion CBR:
  - Longitud: {final_output.get("length", "no indicada")}
  - Nivel tecnico: {final_output.get("technical_level", "no indicado")}
  - Estructura: {final_output.get("structure", "no indicada")}
  - Formato: {final_output.get("preferred_format", "no indicado")}
- Descripcion XAI disponible: {xai_description or "No hay descripcion XAI precalculada."}
{change_block}
Instrucciones de salida:
- Da SOLO la explicacion final util para el usuario.
- Usa la respuesta/preferencia recuperada por CBR para personalizar longitud, tono, estructura y nivel tecnico.
- No incluyas apartados de descripcion de la imagen original.
- No incluyas apartados de clasificacion del modelo.
- No incluyas ejemplos de instancias similares, casos vecinos, CBR, IDs, similitudes ni rankings.
- No incluyas contraejemplos.
- No incluyas limitaciones, dudas ni falta de evidencia.
- Centrate en que zonas o atributos resalta el metodo XAI y por que apoyan la clase predicha.
- Si falta informacion concreta sobre zonas resaltadas, explica el metodo con cautela sin inventar detalles.
- Usa un texto breve, claro y directo, de 2 a 4 parrafos o 3 puntos como maximo.
- Termina con una frase breve preguntando si la explicacion le convence.
"""

def method_to_option(method):
    for option, option_method in OPTION_TO_METHOD.items():
        if option_method == method and "Opción" in option:
            return option
    for option, option_method in OPTION_TO_METHOD.items():
        if option_method == method:
            return option
    return None


def select_next_distinct_solution(ranking, rejected_methods=None, preferred_method=None):
    rejected_methods = set(rejected_methods or [])
    if preferred_method and preferred_method not in rejected_methods:
        preferred_option = method_to_option(preferred_method)
        if preferred_option is not None:
            return preferred_option, preferred_method
    for option in ranking["selected_option"]:
        option_text = str(option)
        method = OPTION_TO_METHOD.get(option_text)
        if method is None:
            continue
        if method not in rejected_methods:
            return option_text, method
    return None, None

def retrieve_neighbors_with_fallback(query_real):
    attempts = [
        (SAME_IMAGE_ONLY, MIN_SIMILARITY, "umbral configurado"),
        (SAME_IMAGE_ONLY, None, "sin umbral minimo"),
        (False, None, "sin umbral y permitiendo otras imagenes"),
    ]
    for same_image_only, min_similarity, label in attempts:
        neighbors = retriever.get_neighbors(
            query_real,
            k=ALTERNATIVE_POOL_SIZE,
            same_image_only=same_image_only,
            min_similarity=min_similarity,
        )
        if not neighbors.empty:
            note = (
                f"Recuperacion usada: {label}. "
                f"same_image_only={same_image_only}, min_similarity={min_similarity}."
            )
            return neighbors, note
    raise RuntimeError("No se encontraron vecinos en la base de casos.")


def retrieve_counterexamples(query_real, k=2):
    candidates = retriever.get_neighbors(
        query_real,
        k=ALTERNATIVE_POOL_SIZE,
        same_image_only=False,
        min_similarity=None,
    ).copy()
    if candidates.empty:
        return candidates
    query_image_id = query_real.get("image_id")
    query_class = str(query_real.get("model_predicted_class", "")).strip().lower()
    candidates = candidates[candidates["image_id"] != query_image_id]
    if query_class and "model_predicted_class" in candidates.columns:
        candidates = candidates[
            candidates["model_predicted_class"].fillna("").astype(str).str.strip().str.lower() != query_class
        ]
    if candidates.empty:
        return candidates
    return candidates.sort_values("similarity", ascending=True).head(k).copy()


def counterexamples_to_records(counterexamples):
    records = []
    for _, row in counterexamples.iterrows():
        record = {
            "case_id": str(row.get("case_id", "")),
            "user_id": str(row.get("user_id", "")),
            "image_id": int(row.get("image_id")) if pd.notna(row.get("image_id")) else None,
            "selected_option": str(row.get("selected_option", "")),
            "model_predicted_class": str(row.get("model_predicted_class", "")),
            "similarity": round(float(row.get("similarity", 0.0)), 3),
            "initial_description": str(row.get("initial_description", "")),
        }
        records.append(record)
    return records


def generate_interactive_explanation(requested_change=None, rejected_methods=None, preferred_method=None):
    query_real = retriever.enrich_query_with_image_metadata(build_interactive_query())
    neighbors_pool_real, retrieval_note = retrieve_neighbors_with_fallback(query_real)
    interactive_state["retrieval_note"] = retrieval_note
    ranking_pool_real = retriever.recommend_explanation(neighbors_pool_real)
    option_real, method_real = select_next_distinct_solution(
        ranking_pool_real,
        rejected_methods=rejected_methods,
        preferred_method=preferred_method,
    )
    if option_real is None or method_real is None:
        return None, neighbors_pool_real.head(K_NEIGHBORS).copy(), ranking_pool_real
    neighbor_real = retriever.best_neighbor_for_option(neighbors_pool_real, option_real)
    query_real = complete_query_with_recovered_preferences(query_real, neighbor_real)
    neighbors_real = neighbors_pool_real.head(K_NEIGHBORS).copy()
    if option_real not in set(neighbors_real["selected_option"].astype(str)):
        alternative_row = neighbors_pool_real[neighbors_pool_real["selected_option"].astype(str) == option_real].head(1)
        neighbors_real = pd.concat([neighbors_real, alternative_row], ignore_index=True).drop_duplicates("case_id")
    ranking_real = ranking_pool_real
    counterexamples_real = retrieve_counterexamples(query_real, k=2)
    counterexample_records = counterexamples_to_records(counterexamples_real)
    interactive_state["counterexamples"] = counterexamples_real
    xai_text_real = get_xai_description(
        DESCRIPTIONS,
        method=method_real,
        description_case_id=IMAGE_ID_TO_DESCRIPTION_CASE[query_real["image_id"]],
    )
    prompt_real = build_final_user_prompt(
        query=query_real,
        recommended_option=option_real,
        recommended_method=method_real,
        xai_description=xai_text_real,
        neighbor=neighbor_real,
        requested_change=requested_change,
    )
    problem_real = build_problem_representation(query_real)
    solution_real = build_solution_representation(query_real, option_real, method_real)
    if use_ollama_w.value:
        explanation_real = generate_with_ollama(prompt_real, OLLAMA_MODEL)
    else:
        explanation_real = fallback_explanation(METHOD_LABEL[method_real], xai_text_real)
    interactive_state["iteration"] += 1
    result_real = PipelineResult(
        timestamp=pd.Timestamp.now().isoformat(timespec="seconds"),
        query=query_real,
        recommended_option=option_real,
        recommended_method=method_real,
        neighbor_case_id=str(neighbor_real["case_id"]),
        neighbor_similarity=float(neighbor_real["similarity"]),
        xai_description=xai_text_real,
        prompt=prompt_real,
        explanation=explanation_real,
        problem=problem_real,
        solution=solution_real,
        counterexamples=counterexample_records,
        requested_change=requested_change,
        iteration=interactive_state["iteration"],
    )
    interactive_state["result"] = result_real
    interactive_state["neighbor"] = neighbor_real
    return result_real, neighbors_real, ranking_real

def record_generation(result, neighbors, ranking, requested_change=None):
    neighbor_cols = ["case_id", "user_id", "image_id", "selected_option", "similarity", "mean_helpfulness"]
    interactive_state["session_history"].append({
        "event": "generation",
        "iteration": result.iteration,
        "requested_change": requested_change,
        "result": result.__dict__.copy(),
        "neighbors": neighbors[neighbor_cols].copy(),
        "ranking": ranking.copy(),
    })

def record_feedback(result):
    interactive_state["session_history"].append({
        "event": "feedback",
        "iteration": result.iteration,
        "convinced": result.convinced,
        "rating": result.rating,
        "acceptance": result.acceptance,
        "satisfaction": result.satisfaction,
        "confidence": result.confidence,
        "understanding": result.understanding,
        "requested_change": result.requested_change,
    })

def render_final_explanation(result):
    description_case_id = IMAGE_ID_TO_DESCRIPTION_CASE.get(result.query.get("image_id"))
    xai_widgets = []
    if description_case_id is not None:
        xai_widgets = build_xai_image_widgets(
            description_case_id,
            result.recommended_method,
            title="Metodo XAI mostrado",
        )
    return (
        html_title("Explicacion generada", level=3),
        *xai_widgets,
        html_text(result.explanation),
    )

def append_new_explanation(result, neighbors):
    result_box.children = render_final_explanation(result)

def render_feedback_controls(message=None, free_feedback_mode=False):
    convinced_w = widgets.ToggleButtons(
        options=[("Si", True), ("No", False)],
        description="Convence?",
        value=True,
    )
    acceptance_w = widgets.IntSlider(description="Aceptacion", min=1, max=5, value=4)
    satisfaction_w = widgets.IntSlider(description="Satisfaccion", min=1, max=5, value=4)
    confidence_w = widgets.IntSlider(description="Confianza", min=1, max=5, value=4)
    understanding_w = widgets.IntSlider(description="Comprension", min=1, max=5, value=4)
    positive_feedback_box = widgets.VBox([
        widgets.HTML("<b>Evalua la explicacion de 1 a 5</b>"),
        acceptance_w,
        satisfaction_w,
        confidence_w,
        understanding_w,
    ])
    needs_w = widgets.Textarea(
        description="Necesito",
        placeholder="Que necesitas saber o en que formato lo quieres?",
        layout=widgets.Layout(width="850px", height="80px"),
    )
    submit_w = widgets.Button(description="Enviar feedback", button_style="success")
    status_w = widgets.HTML("")

    def refresh_visible_fields(change=None):
        if convinced_w.value:
            positive_feedback_box.layout.display = ""
            needs_w.layout.display = "none"
        else:
            positive_feedback_box.layout.display = "none"
            needs_w.layout.display = "" if free_feedback_mode else "none"

    def save_negative_feedback(current, requested_change=None, alternatives_exhausted=False):
        current.convinced = False
        current.rating = None
        current.acceptance = None
        current.satisfaction = None
        current.confidence = None
        current.understanding = None
        current.requested_change = requested_change
        current.alternatives_exhausted = alternatives_exhausted
        record_feedback(current)
        save_result(current, OUTPUT_DIR)

    def on_submit(_):
        current = interactive_state["result"]
        if current is None:
            return

        if convinced_w.value:
            current.convinced = True
            current.acceptance = int(acceptance_w.value)
            current.satisfaction = int(satisfaction_w.value)
            current.confidence = int(confidence_w.value)
            current.understanding = int(understanding_w.value)
            current.rating = int(round((
                current.acceptance + current.satisfaction + current.confidence + current.understanding
            ) / 4))
            current.requested_change = None
            current.alternatives_exhausted = False
            record_feedback(current)
            path = save_result(current, OUTPUT_DIR)
            feedback_box.children = (
                widgets.HTML("Feedback guardado. Experimento completado."),
            )
            return

        rejected_methods = interactive_state.setdefault("rejected_methods", [])
        if current.recommended_method not in rejected_methods:
            rejected_methods.append(current.recommended_method)

        if not free_feedback_mode:
            save_negative_feedback(current, requested_change=None, alternatives_exhausted=False)
            status_w.value = "Generando una nueva explicacion..."
            new_result, new_neighbors, new_ranking = generate_interactive_explanation(
                requested_change=None,
                rejected_methods=rejected_methods,
            )
            if new_result is None:
                feedback_box.children = (
                    render_feedback_controls(
                        "Se han agotado los metodos XAI alternativos del ranking CBR. Ahora indica que necesitas saber o en que formato lo quieres:",
                        free_feedback_mode=True,
                    ),
                )
                return
            record_generation(new_result, new_neighbors, new_ranking, requested_change=None)
            append_new_explanation(new_result, new_neighbors)
            feedback_box.children = (
                render_feedback_controls(
                    "La respuesta anterior se guardo como no convincente. Evalua la nueva explicacion:"
                ),
            )
            return

        requested_change = needs_w.value.strip()
        if not requested_change:
            status_w.value = "<span style='color:#b00020'>Indica que necesitas saber o en que formato lo quieres.</span>"
            return

        save_negative_feedback(current, requested_change=requested_change, alternatives_exhausted=True)
        feedback_box.children = (
            widgets.HTML("Feedback final guardado. Se han agotado los metodos XAI y el experimento queda cerrado."),
        )
        return

    convinced_w.observe(refresh_visible_fields, names="value")
    submit_w.on_click(on_submit)
    refresh_visible_fields()

    items = []
    if message:
        items.append(widgets.HTML(f"<b>{html.escape(message)}</b>"))
    items.extend([convinced_w, positive_feedback_box])
    if free_feedback_mode:
        items.append(needs_w)
    items.extend([submit_w, status_w])
    return widgets.VBox(items)

def on_generate(_):
    if interactive_state.get("is_generating"):
        return
    interactive_state["is_generating"] = True
    generate_button.disabled = True
    try:
        interactive_state["iteration"] = 0
        interactive_state["session_history"] = []
        interactive_state["rejected_methods"] = []
        result_real, neighbors_real, ranking_real = generate_interactive_explanation()
        record_generation(result_real, neighbors_real, ranking_real)
        result_box.children = render_final_explanation(result_real)
        feedback_box.children = (render_feedback_controls(),)
    finally:
        generate_button.disabled = False
        interactive_state["is_generating"] = False

# Evita duplicados si la celda se ejecuta varias veces en la misma sesion.
generate_button._click_handlers.callbacks = []
generate_button.on_click(on_generate)


### Perfil e imagen

VBox()

VBox()